<a href="https://colab.research.google.com/github/aryakamat01-sketch/pbs-generic-erosion/blob/main/pbs_generic_erosion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
print(pd.__version__)

2.2.3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

path = '/content/drive/MyDrive/pbs-data/dos-jul-2024-to-jun-2025-phrmcy-type.csv'
df = pd.read_csv(path)

print(df.shape)
print(df.columns.tolist())
df.head()

(389050, 12)
['MONTH_OF_SUPPLY', 'ITEM_CODE', 'DRUG_TYPE', 'PATIENT_CAT', 'PHRMCY_TYPE', 'SCRIPT_TYPE', 'PRESCRIPTIONS', 'PATIENT_CONTRIB', 'GOVT_CONTRIB', 'TOTAL_COST', 'RETAIL_MARKUP', 'PATIENT_NET_CONTRIB']


,MONTH_OF_SUPPLY,ITEM_CODE,DRUG_TYPE,PATIENT_CAT,PHRMCY_TYPE,SCRIPT_TYPE,PRESCRIPTIONS,PATIENT_CONTRIB,GOVT_CONTRIB,TOTAL_COST,RETAIL_MARKUP,PATIENT_NET_CONTRIB
0,202407,00000B,UK,R0,S90,ABOVE CO-PAYMENT,1378,7.7,86293.70,86301.40,0.0,23.10
1,202407,00000B,UK,R0,S90,UNDER CO-PAYMENT,33,0.0,0.00,0.00,0.0,437.96
2,202407,00000B,UK,R0,S94 PRI,ABOVE CO-PAYMENT,1,0.0,89.40,89.40,0.0,0.00
3,202407,00000B,UK,R0,S94 PUB,ABOVE CO-PAYMENT,1,0.0,6.50,6.50,0.0,0.00
4,202407,00000B,UK,R1,S90,ABOVE CO-PAYMENT,922,6968.5,113936.68,120905.18,0.0,4860.59


In [4]:
print(df.columns.tolist())
print(df['SCRIPT_TYPE'].value_counts())
print(df['DRUG_TYPE'].value_counts())
print(df['PATIENT_CAT'].value_counts())
print(df['PHRMCY_TYPE'].value_counts())

['MONTH_OF_SUPPLY', 'ITEM_CODE', 'DRUG_TYPE', 'PATIENT_CAT', 'PHRMCY_TYPE', 'SCRIPT_TYPE', 'PRESCRIPTIONS', 'PATIENT_CONTRIB', 'GOVT_CONTRIB', 'TOTAL_COST', 'RETAIL_MARKUP', 'PATIENT_NET_CONTRIB']
SCRIPT_TYPE
ABOVE CO-PAYMENT    325103
UNDER CO-PAYMENT     63947
Name: count, dtype: int64
DRUG_TYPE
GE    362061
PL     11509
R1      6175
DT      3969
OT      3138
EP      1568
DB       476
RN        79
UK        75
Name: count, dtype: int64
PATIENT_CAT
G2    118457
C1     99660
C0     64780
R1     45702
R0     35109
G1     24866
DB       476
Name: count, dtype: int64
PHRMCY_TYPE
S90        234374
S94 PUB     79530
S94 PRI     65845
S92          9301
Name: count, dtype: int64


In [5]:
work = df[(df['SCRIPT_TYPE'] == 'ABOVE CO-PAYMENT') & (df['PHRMCY_TYPE'] == 'S90')].copy()
print(work.shape)

(210612, 12)


In [6]:
print(work['PRESCRIPTIONS'].describe())
print((work['PRESCRIPTIONS'] <= 0).sum())
print((work['TOTAL_COST'] <= 0).sum())

count    210612.000000
mean       1075.260645
std        6224.981539
min           1.000000
25%           5.000000
50%          31.000000
75%         239.000000
max      301091.000000
Name: PRESCRIPTIONS, dtype: float64
0
0


In [8]:
drug_map = pd.read_csv('/content/drive/MyDrive/pbs-data/pbs-item-drug-map.csv', encoding='latin-1')
print(drug_map.shape)
drug_map.head()

(12162, 4)


,ITEM_CODE,DRUG_NAME,FORM/STRENGTH,ATC5_Code
0,00000A,MISSING ITEM CODE,Missing Item Code,Z
1,00013Q,EXTEMPORANEOUSLY PREPARED,Creams,Z
2,00015T,EXTEMPORANEOUSLY PREPARED,Ear drops,Z
3,00016W,ELIXIRS,Generic term,Z
4,00019B,EXTEMPORANEOUSLY PREPARED,Eye drops containing cocaine hcl,Z


In [9]:
work = work.merge(drug_map, on='ITEM_CODE', how='left')
print(work.shape)
print(work['DRUG_NAME'].isna().sum())
print(work['DRUG_NAME'].nunique())

(210612, 15)
48
925


In [10]:
work['date'] = pd.to_datetime(work['MONTH_OF_SUPPLY'], format='%Y%m')
work['cost_per_script'] = work['TOTAL_COST'] / work['PRESCRIPTIONS']

print(work['date'].min(), work['date'].max())
print(work['cost_per_script'].describe())

2024-07-01 00:00:00 2025-06-01 00:00:00
count    210612.000000
mean        558.740462
std        1814.333197
min           8.945000
25%          23.030000
50%          46.127332
75%         204.830000
max       44013.210000
Name: cost_per_script, dtype: float64


In [11]:
import glob

files = sorted(glob.glob('/content/drive/MyDrive/pbs-data/dos-*.csv'))
print(files)

frames = [pd.read_csv(f) for f in files]
raw = pd.concat(frames, ignore_index=True)
print(raw.shape)

['/content/drive/MyDrive/pbs-data/dos-jul-2022-to-jun-2023-phrmcy-type.csv', '/content/drive/MyDrive/pbs-data/dos-jul-2023-to-jun-2024-phrmcy-type.csv', '/content/drive/MyDrive/pbs-data/dos-jul-2024-to-jun-2025-phrmcy-type.csv', '/content/drive/MyDrive/pbs-data/dos-jul-2025-to-jun-2026-phrmcy-type.csv']
(1490524, 12)


In [12]:
work = raw[(raw['SCRIPT_TYPE'] == 'ABOVE CO-PAYMENT') & (raw['PHRMCY_TYPE'] == 'S90')].copy()
work = work.merge(drug_map, on='ITEM_CODE', how='left')
work['date'] = pd.to_datetime(work['MONTH_OF_SUPPLY'], format='%Y%m')
work['cost_per_script'] = work['TOTAL_COST'] / work['PRESCRIPTIONS']

print(work.shape)
print(work['date'].min(), work['date'].max())
print(work['date'].nunique())

(789577, 17)
2022-07-01 00:00:00 2026-06-01 00:00:00
48


In [13]:
work.groupby('date')['PRESCRIPTIONS'].sum().tail(6)

,PRESCRIPTIONS
date,
2026-01-01,17255493
2026-02-01,17553355
2026-03-01,19947341
2026-04-01,18935944
2026-05-01,19676963
2026-06-01,19992133
